In [ ]:
import torch
import shutil
import numpy as np
import cv2
import os
import subprocess
from pathlib import Path
from models.DepthAnythingV2.depth_anything_v2.dpt import DepthAnythingV2

## Depth Anything V2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f'selected device: {device}')

In [ ]:
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vitl' # or 'vits', 'vitb', 'vitg'

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(f'models/DepthAnythingV2/checkpoints/depth_anything_v2_{encoder}.pth', map_location='cpu'))
model = model.to(device).eval()

# COLMAP

https://github.com/colmap/colmap?tab=readme-ov-file

In [ ]:
COLMAP_PATH = r"C:\Program Files\colmap-x64-windows-cuda\bin\colmap.exe"

def run_colmap_cmd(*args):
    subprocess.run([COLMAP_PATH] + list(args), check=True)

def run_colmap_feature_extractor(db, imgs):
    run_colmap_cmd("feature_extractor", "--database_path", db, "--image_path", imgs, "--ImageReader.camera_model", "PINHOLE", "--ImageReader.single_camera", "1")

def run_colmap_exhaustive_matcher(db):
    run_colmap_cmd("exhaustive_matcher", "--database_path", db)

def run_colmap_mapper(db, imgs, out):
    os.makedirs(out, exist_ok=True)
    run_colmap_cmd("mapper", "--database_path", db, "--image_path", imgs, "--output_path", out)

def run_colmap_model_converter(inp, out, typ="TXT"):
    run_colmap_cmd("model_converter", "--input_path", inp, "--output_path", out, "--output_type", typ)

# Paths
colmap_dir = Path(r"C:\Github_FabianDubach\HSLU.DSPRO2.Beyond2D\colmap")
images_dir = colmap_dir / "images"
database_path = colmap_dir / "database.db"
sparse_output_path = colmap_dir / "sparse"
sparse_txt_path = colmap_dir / "sparse-txt"

# Pipeline
run_colmap_feature_extractor(str(database_path), str(images_dir))
run_colmap_exhaustive_matcher(str(database_path))
run_colmap_mapper(str(database_path), str(images_dir), str(sparse_output_path))
if sparse_txt_path.exists():
    shutil.rmtree(sparse_txt_path)  # Delete existing output folder
ply_output_path = colmap_dir / "model.ply"
run_colmap_model_converter(str(sparse_output_path / "0"), str(ply_output_path), typ="PLY")

In [ ]:
def run_colmap_image_undistorter(imgs, sparse_model, dense_output):
    os.makedirs(dense_output, exist_ok=True)
    run_colmap_cmd(
        "image_undistorter",
        "--image_path", imgs,
        "--input_path", sparse_model,
        "--output_path", dense_output,
        "--output_type", "COLMAP",
        "--max_image_size", "2000"
    )

In [ ]:
def run_colmap_patch_match_stereo(dense_output):
    run_colmap_cmd(
        "patch_match_stereo",
        "--workspace_path", dense_output,
        "--workspace_format", "COLMAP",
        "--PatchMatchStereo.geom_consistency", "true"
    )

In [ ]:
def run_colmap_stereo_fusion(dense_output, output_ply_path):
    run_colmap_cmd(
        "stereo_fusion",
        "--workspace_path", dense_output,
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", output_ply_path
    )

In [ ]:
dense_output_path = colmap_dir / "dense"
dense_ply_output = colmap_dir / "dense.ply"

run_colmap_image_undistorter(str(images_dir), str(sparse_output_path / "0"), str(dense_output_path))
run_colmap_patch_match_stereo(str(dense_output_path))
run_colmap_stereo_fusion(str(dense_output_path), str(dense_ply_output))

# COLMAP with Depth-Anything-V2

In [ ]:
import torch
import subprocess
import os
import shutil
from pathlib import Path
import numpy as np
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from models.DepthAnythingV2.depth_anything_v2.dpt import DepthAnythingV2

# Load model
checkpoint_path = r"./models/DepthAnythingV2/checkpoints/depth_anything_v2_vitl.pth"
model = DepthAnythingV2()
state_dict = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(state_dict)
model.eval().cuda()

COLMAP_PATH = r"C:\Program Files\colmap-x64-windows-cuda\bin\colmap.exe"

def run_colmap_cmd(*args):
    subprocess.run([COLMAP_PATH] + list(args), check=True)

def run_colmap_feature_extractor(db, imgs):
    run_colmap_cmd(
        "feature_extractor",
        "--database_path", db,
        "--image_path", imgs,
        "--ImageReader.camera_model", "PINHOLE",
        "--ImageReader.single_camera", "1",
        "--ImageReader.default_focal_length_factor", "1.2"
    )


def run_colmap_sequential_matcher(db, overlap=5):
    run_colmap_cmd(
        "sequential_matcher",
        "--database_path", db,
        "--SequentialMatching.overlap", str(overlap)
    )


def run_colmap_mapper(db, imgs, out):
    os.makedirs(out, exist_ok=True)
    run_colmap_cmd("mapper", "--database_path", db, "--image_path", imgs, "--output_path", out)

def run_colmap_model_converter(inp, out, typ="TXT"):
    run_colmap_cmd("model_converter", "--input_path", inp, "--output_path", out, "--output_type", typ)

def run_colmap_image_undistorter(imgs, sparse_model, dense_output):
    os.makedirs(dense_output, exist_ok=True)
    run_colmap_cmd(
        "image_undistorter",
        "--image_path", imgs,
        "--input_path", sparse_model,
        "--output_path", dense_output,
        "--output_type", "COLMAP",
        "--max_image_size", "2000"
    )

def run_colmap_patch_match_stereo(dense_output):
    """Run COLMAP's patch match stereo to generate depth and normal maps"""
    run_colmap_cmd(
        "patch_match_stereo",
        "--workspace_path", dense_output,
        "--workspace_format", "COLMAP",
        "--PatchMatchStereo.geom_consistency", "true"
    )

def run_colmap_stereo_fusion(dense_output, output_ply_path):
    run_colmap_cmd(
        "stereo_fusion",
        "--workspace_path", dense_output,
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",
        "--output_path", output_ply_path
    )

def save_colmap_depth_map(depth_array, output_path):
    """Save depth map in COLMAP's proper binary format"""
    height, width = depth_array.shape
    
    with open(output_path, 'wb') as f:
        # Write header in the correct COLMAP format: width&height&channels&
        header = f"{width}&{height}&{1}&"
        f.write(header.encode('utf-8'))
        
        # Write depth data as float32 in row-major order
        depth_array.astype(np.float32).tofile(f)

def process_depth_with_depth_anything_v2():
    """Generate depth maps using Depth Anything V2"""
    transform = Compose([
        Resize((518, 518)),  # Use larger input size for better quality
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
    ])

    input_dir = colmap_dir / "dense" / "images"
    # Create separate directory for AI-generated depth maps
    ai_depth_maps_dir = colmap_dir / "dense" / "stereo" / "depth_maps_ai"
    os.makedirs(ai_depth_maps_dir, exist_ok=True)

    print("Generating depth maps with Depth Anything V2...")
    
    processed_images = []
    for img_name in os.listdir(input_dir):
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
            
        print(f"Processing {img_name}...")
        img_path = input_dir / img_name
        img = Image.open(img_path).convert("RGB")
        original_size = img.size
        
        # Prepare input
        img_input = transform(img).unsqueeze(0).cuda()
        
        with torch.no_grad():
            # Get depth prediction
            depth = model(img_input)
            depth = depth.squeeze().cpu().numpy()
        
        # Resize back to original image size
        depth_resized = np.array(Image.fromarray(depth).resize(original_size, Image.BILINEAR))
        
        # Convert relative depth to metric depth (this is crucial!)
        # Depth Anything V2 outputs relative depths, we need to scale them appropriately
        min_depth = 0.1  # 10cm minimum depth
        max_depth = 20.0  # 20m maximum depth - more realistic for most scenes
        
        # Normalize and scale depth properly
        depth_min, depth_max = depth_resized.min(), depth_resized.max()
        if depth_max > depth_min:  # Avoid division by zero
            # Normalize to [0, 1]
            depth_normalized = (depth_resized - depth_min) / (depth_max - depth_min)
            # Depth Anything V2 might need inversion depending on the model
            # Try both with and without inversion to see which works better
            # depth_normalized = 1.0 - depth_normalized  # Uncomment if depths are inverted
            depth_metric = min_depth + depth_normalized * (max_depth - min_depth)
        else:
            depth_metric = np.full_like(depth_resized, min_depth)
        
        # Ensure no invalid depth values
        depth_metric = np.clip(depth_metric, min_depth, max_depth)
        
        # Filter out invalid pixels (you may need to adjust this)
        valid_mask = (depth_metric > min_depth) & (depth_metric < max_depth)
        if not valid_mask.any():
            print(f"WARNING: No valid depths found for {img_name}")
            continue
        
        # Save in COLMAP geometric format
        depth_filename = img_name.rsplit('.', 1)[0] + ".geometric.bin"
        depth_path = ai_depth_maps_dir / depth_filename
        save_colmap_depth_map(depth_metric, depth_path)
        
        print(f"Saved AI depth map: {depth_filename}")
        print(f"  Depth range: {depth_metric.min():.2f}m to {depth_metric.max():.2f}m")
        processed_images.append(img_name)
    
    return processed_images

def copy_ai_depths_to_stereo_folder():
    """Copy AI-generated depth maps to the stereo folder, replacing COLMAP's"""
    ai_depth_dir = colmap_dir / "dense" / "stereo" / "depth_maps_ai"
    colmap_depth_dir = colmap_dir / "dense" / "stereo" / "depth_maps"
    
    if not ai_depth_dir.exists():
        print("No AI depth maps found!")
        return
    
    # Ensure the stereo depth_maps directory exists
    os.makedirs(colmap_depth_dir, exist_ok=True)
    
    # Copy AI depth maps to replace COLMAP's
    for depth_file in ai_depth_dir.glob("*.geometric.bin"):
        dest_file = colmap_depth_dir / depth_file.name
        shutil.copy2(depth_file, dest_file)
        print(f"Copied AI depth map: {depth_file.name}")

def run_stereo_fusion_with_ai_depths():
    """Run stereo fusion using only the AI-generated depth maps"""
    print("Using AI-generated depth maps for fusion...")
    run_colmap_cmd(
        "stereo_fusion",
        "--workspace_path", str(dense_output_path),
        "--workspace_format", "COLMAP",
        "--input_type", "geometric",  # Use geometric depth maps
        "--output_path", str(dense_ply_output),
        "--StereoFusion.min_num_pixels", "3",  # Lower threshold for fusion
        "--StereoFusion.max_reproj_error", "8.0",  # More lenient reprojection error
        "--StereoFusion.max_depth_error", "0.2",   # More lenient depth error
        "--StereoFusion.max_normal_error", "20",   # More lenient normal error
        "--StereoFusion.check_num_images", "3"  # Add this parameter
    )

def debug_depth_maps():
    """Debug function to verify depth maps are correctly formatted"""
    depth_dir = colmap_dir / "dense" / "stereo" / "depth_maps"
    if not depth_dir.exists():
        print("No depth maps directory found!")
        return
    
    for depth_file in depth_dir.glob("*.geometric.bin"):
        print(f"Checking {depth_file.name}...")
        try:
            with open(depth_file, 'rb') as f:
                # Read header
                header_data = f.read(100)  # Read enough to get header
                header_str = header_data.decode('utf-8', errors='ignore')
                print(f"  Header: {header_str[:50]}...")
                
                # Find the end of header (after last &)
                header_end = header_str.rfind('&') + 1
                if header_end > 0:
                    dimensions = header_str[:header_end-1].split('&')
                    if len(dimensions) >= 3:
                        width, height, channels = map(int, dimensions[:3])
                        print(f"  Dimensions: {width}x{height}, channels: {channels}")
                        
                        # Check file size
                        expected_size = header_end + width * height * 4  # 4 bytes per float32
                        actual_size = depth_file.stat().st_size
                        print(f"  Expected size: {expected_size}, Actual size: {actual_size}")
                        
                        if abs(expected_size - actual_size) > 100:  # Allow some tolerance
                            print(f"  WARNING: Size mismatch!")
                    else:
                        print(f"  ERROR: Invalid header format")
        except Exception as e:
            print(f"  ERROR reading file: {e}")

# Main pipeline
colmap_dir = Path(r"./colmap")
images_dir = colmap_dir / "images//cabinet"
database_path = colmap_dir / "database.db"
sparse_output_path = colmap_dir / "sparse"
dense_output_path = colmap_dir / "dense"
dense_ply_output = colmap_dir / "dense_model.ply"

print("Running COLMAP sparse reconstruction...")
run_colmap_feature_extractor(str(database_path), str(images_dir))
run_colmap_sequential_matcher(str(database_path), overlap=5)  # Adjust overlap as needed
run_colmap_mapper(str(database_path), str(images_dir), str(sparse_output_path))

print("Converting sparse model to PLY...")
ply_output_path = colmap_dir / "sparse_model.ply"
run_colmap_model_converter(str(sparse_output_path / "0"), str(ply_output_path), typ="PLY")

print("Undistorting images...")
run_colmap_image_undistorter(str(images_dir), str(sparse_output_path / "0"), str(dense_output_path))

print("Generating depth maps with Depth Anything V2...")
processed_images = process_depth_with_depth_anything_v2()

# DO NOT run patch match stereo - we want to use only AI depths
print("Running COLMAP patch match stereo to generate normal maps...")
run_colmap_patch_match_stereo(str(dense_output_path))

print("Copying AI depth maps to stereo folder...")
copy_ai_depths_to_stereo_folder()

print("Debugging depth map format...")
debug_depth_maps()

print("Fusing AI depth maps into dense point cloud...")
run_stereo_fusion_with_ai_depths()

print(f"Dense reconstruction complete! Output saved to: {dense_ply_output}")
print("This point cloud was generated using Depth Anything V2 depth maps!")

# Optional: Create a comparison by also running traditional COLMAP
print("\n--- Creating comparison with traditional COLMAP ---")
traditional_output = colmap_dir / "dense_traditional.ply"
print("Running traditional COLMAP patch match stereo for comparison...")
run_colmap_patch_match_stereo(str(dense_output_path))
print("Fusing traditional COLMAP depth maps...")
run_colmap_cmd(
    "stereo_fusion",
    "--workspace_path", str(dense_output_path),
    "--workspace_format", "COLMAP", 
    "--input_type", "geometric",
    "--output_path", str(traditional_output)
)
print(f"Traditional COLMAP result saved to: {traditional_output}")
print("Now you can compare the AI-generated vs traditional results!")

xFormers not available
xFormers not available


Running COLMAP sparse reconstruction...
Converting sparse model to PLY...
Undistorting images...
Generating depth maps with Depth Anything V2...
Generating depth maps with Depth Anything V2...
Processing 00281.jpg...
Saved AI depth map: 00281.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00284.jpg...
Saved AI depth map: 00284.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00285.jpg...
Saved AI depth map: 00285.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00292.jpg...
Saved AI depth map: 00292.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00297.jpg...
Saved AI depth map: 00297.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00303.jpg...
Saved AI depth map: 00303.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00307.jpg...
Saved AI depth map: 00307.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00312.jpg...
Saved AI depth map: 00312.geometric.bin
  Depth range: 0.10m to 20.00m
Processing 00447.jpg...
Saved AI depth map: 004

In [ ]:
def filter_point_cloud(input_ply, output_ply):
    """Filter the point cloud with gentler parameters to preserve structure"""
    import open3d as o3d
    import numpy as np
    
    print(f"Filtering point cloud with gentle parameters: {input_ply}")
    pcd = o3d.io.read_point_cloud(str(input_ply))
    
    # Get number of points before filtering
    n_points_before = len(pcd.points)
    print(f"Points before filtering: {n_points_before}")
    
    # Statistical outlier removal - much gentler
    print("Applying gentle statistical outlier removal...")
    pcd, _ = pcd.remove_statistical_outlier(
        nb_neighbors=50,    # Consider more neighbors (was 20)
        std_ratio=3.0       # More permissive threshold (was 2.0)
    )
    
    # Radius outlier removal - much gentler
    print("Applying gentle radius outlier removal...")
    pcd, _ = pcd.remove_radius_outlier(
        nb_points=2,        # Require fewer points (was 16)
        radius=0.2          # Larger radius (was 0.05)
    )
    
    # Get number of points after filtering
    n_points_after = len(pcd.points)
    print(f"Points after filtering: {n_points_after}")
    print(f"Removed {n_points_before - n_points_after} points ({(1 - n_points_after/n_points_before)*100:.1f}%)")
    
    # Save the filtered point cloud
    o3d.io.write_point_cloud(str(output_ply), pcd)
    print(f"Saved filtered point cloud to: {output_ply}")
    
    return output_ply

In [ ]:
# After stereo fusion
filtered_ply_output = colmap_dir / "dense_model_filtered.ply"
filter_point_cloud(dense_ply_output, filtered_ply_output)